# NeoNatal Watch AI — Phase 8: Transformer

> **DEMO / SYNTHETIC DATA — NOT FOR CLINICAL USE**
>
> This notebook builds our final Deep Learning model: a Time-Series Transformer using Self-Attention.

---

## Why Transformers?
- LSTMs read data left-to-right. They can sometimes 'forget' important events that happened at the beginning of the 30-minute window.
- Transformers look at all 30 minutes **simultaneously**. 
- The **Self-Attention** mechanism calculates how much 'attention' the model should pay to minute 5 vs minute 25 when predicting a deterioration event at minute 30.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import tensorflow as tf
from ml.models.transformer import build_transformer_model
from ml.evaluation.metrics import evaluate_model

print('Imports OK')

In [ ]:
# Load 3D tensors
X_train = np.load('../data/processed/X_train.npy')
y_train = np.load('../data/processed/y_train.npy')
X_val = np.load('../data/processed/X_val.npy')
y_val = np.load('../data/processed/y_val.npy')
X_test = np.load('../data/processed/X_test.npy')
y_test = np.load('../data/processed/y_test.npy')

In [ ]:
# Build the Transformer Architecture
model = build_transformer_model(
    sequence_length=X_train.shape[1],
    n_features=X_train.shape[2],
    num_transformer_blocks=2,
    head_size=32,
    num_heads=4
)

model.summary()

In [ ]:
# Train the Model (Reduced epochs for demonstration)
n_pos = y_train.sum()
n_neg = len(y_train) - n_pos
class_weight = {0: 1.0, 1: min(n_neg / n_pos, 50.0) if n_pos > 0 else 1.0}

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_pr_auc", mode="max", patience=5, restore_best_weights=True
    )
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=5,
    batch_size=64,
    class_weight=class_weight,
    callbacks=callbacks
)

In [ ]:
# Evaluate on the Test Set
y_test_probs = model.predict(X_test).flatten()

metrics = evaluate_model(
    y_true=y_test,
    y_probs=y_test_probs,
    model_name="Transformer_Notebook",
    save_dir="../reports/figures"
)